# P7 / 01: capitalisation ranks, rank-space residuals and rolling network fits

Protocol: [`001_claude_p7_rank_space_preregistration.md`](../../discussions/Rank_Space/001_claude_p7_rank_space_preregistration.md).

- **Capitalisation:** split-adjusted (not dividend-adjusted) price × point-in-time SEC shares outstanding, on today's split basis.
- **Ranks:** 1–400 each close.
- **Rank returns:** the change in the k-th largest cap. These are not tradable.
- **Residuals:** PCA with 1 factor.
- **Network:** P6's CNN + Transformer.
- **Two arms:** `RS-paper` (gross mean-variance on rank returns) and `RS-net` (Sharpe of the tradable rank-mapped book net of measured costs).
- **Fits:** rolling 750 / 250 / 125, 3 seeds. Every fit is cached.

In [ ]:
import json, os, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import __version__, p1, p6, p7, s04, s05b
from equity_trading.alpaca import ALPACA_VERSION, download_years, session_schedule
from equity_trading.data import vendor_dir, work_dir

SYNTHETIC = os.environ.get("P7_SYNTHETIC") == "1"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.titleweight": "bold", "font.size": 9})
CFG = p7.P7Config()
ARMS = {"RS-paper": "paper", "RS-net": "net"}
CACHE = work_dir() / "results" / ("p7_synthetic" if SYNTHETIC else "p7") / f"v{p7.P7_VERSION}"
CACHE.mkdir(parents=True, exist_ok=True)
if SYNTHETIC:
    CFG = replace(CFG, n_ranks=80, train_days=300, valid_days=100, first_test="2017-01-03", end="2017-12-29",
                  seeds=(0,), epochs=3, patience=3, n_placebo=50, n_boot=2000)
import torch
from equity_trading import p7_model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else " -- CUDA not available"))
print(f"synthetic={SYNTHETIC} | ranks {CFG.n_ranks} | arms {list(ARMS)} | seeds {CFG.seeds}")

## 1. Returns, split-only prices, shares outstanding, capitalisation, costs

In [ ]:
t0 = time.time()
if SYNTHETIC:
    R, CAP = p7.synthetic_market(n_days=1800, n_stocks=120, start="2011-01-03")
    COST = pd.DataFrame(2e-4, index=R.index, columns=R.columns)
else:
    constituents = pd.read_csv(ROOT / "src" / "equity_trading" / "reference" / "sp500_constituents_20260913.csv")
    STOCKS = sorted(constituents["Symbol"])
    sessions = pd.DatetimeIndex(pd.to_datetime(list(session_schedule(list(range(2011, 2026))).index)))
    # total returns (as P6): Yahoo adjusted before 2016-01-04, Alpaca adjusted after
    YH = p6.load_yahoo_daily(STOCKS, "2005-01-01", "2016-01-06", verbose=False)
    download_years(STOCKS, "1Day", list(range(2016, 2026)), workers=8)
    AC, _, _ = p1.load_daily(STOCKS, list(range(2016, 2026)))
    R = p6.splice_returns(YH, AC, sessions, CFG.splice)
    # split-adjusted, not dividend-adjusted prices (for capitalisation)
    YS = p7.yahoo_split_prices(STOCKS, CFG.data_start, "2016-01-06")
    AS = p7.alpaca_split_daily(STOCKS, list(range(2016, 2026)))
    cut = pd.Timestamp(CFG.splice)
    PX = AS.reindex(index=sessions, columns=R.columns)
    ys = YS.reindex(index=sessions, columns=R.columns)
    PX.loc[sessions < cut] = ys.loc[sessions < cut]
    # shares outstanding (SEC, point in time) on today's split basis
    REC = p7.sec_shares(STOCKS)
    SPL, CUR = p7.yahoo_splits_and_shares(STOCKS)
    SH, DIAG = p7.shares_daily(REC, SPL, CUR, sessions, CFG.unit_band)
    DIAG.to_csv(CACHE / "shares_diagnostics.csv", index=False)
    CAP = (PX * SH.reindex(index=sessions, columns=R.columns))
    R = R.loc[:, CAP.notna().any()]
    CAP = CAP[R.columns]
    qc = vendor_dir("Alpaca") / "quote_samples" / "15Min_w3s"
    sess = s05b.sample_sessions([2017, 2018, 2019], 4, seed=42) + s05b.sample_sessions(list(range(2020, 2026)), 4, seed=42)
    samples = pd.concat([pd.read_parquet(qc / f"{d}.parquet") for d in sess], ignore_index=True)
    COST = p1.daily_cost(R.index, list(R.columns), s05b.cost_model(samples))
    print(f"shares: {len(DIAG)} symbols with SEC counts; filings dropped by the C1 cleaning: {int(DIAG['dropped'].sum())}; "
          f"rescaled by the Yahoo unit check: {DIAG.loc[DIAG.rescaled, 'symbol'].tolist()}")
    late = DIAG[pd.to_datetime(DIAG["first_filed"]) > "2012-06-30"].sort_values("first_filed")
    print(f"{len(late)} symbols whose first share count is after 2012-06 (no cap before):", late["symbol"].tolist()[:40])
valid_caps = CAP.notna().sum(axis=1)
print(f"{R.shape[1]} stocks, {len(R)} sessions; stocks with a cap per year (median):")
display(valid_caps.groupby(valid_caps.index.year).median().to_frame("stocks with cap").T)
print(f"days with >= {CFG.n_ranks} caps: {(valid_caps >= CFG.n_ranks).mean():.1%}; ready in {time.time() - t0:.0f}s")

## 2. Rank-space dataset (cached)

In [ ]:
t0 = time.time()
ds_path = CACHE / f"dataset_p7{p7.P7_VERSION}_K{CFG.n_ranks}.npz"
SCHED = p6.fit_schedule(R.index, CFG)
need_from = max(0, SCHED[0]["train"][0] - CFG.lookback)
if ds_path.exists():
    DS = p7.load_dataset(ds_path)
else:
    DS = p7.build_rank_dataset(R, CAP, COST, CFG, start_row=need_from)
    p7.save_dataset(DS, ds_path)
rows = np.arange(need_from, len(R) - 2)
rr, tr = DS["ret"][rows], DS["tret"][rows]
m = np.isfinite(rr) & np.isfinite(tr)
print(f"dataset {DS['x'].shape}; usable rank-days {DS['ok'][need_from:].mean():.1%}; {time.time() - t0:.0f}s")
print(f"rank return vs tradable return of the same rank: corr {np.corrcoef(rr[m], tr[m])[0, 1]:.3f}, "
      f"s.d. {np.std(rr[m]):.4f} vs {np.std(tr[m]):.4f}")
print(f"{len(SCHED)} fits per arm; first predicts from {SCHED[0]['first_day']}")
# C1 data-quality gate (protocol 002): rank returns must look like returns before any training
ratio_sd = np.std(rr[m]) / np.std(tr[m])
big = np.abs(rr[m]) > 0.25
print(f"rank-return s.d. / tradable s.d. = {ratio_sd:.2f}; share of rank-days with |rank return| > 25%: {big.mean():.4%}")
if not SYNTHETIC:
    worst = np.argsort(-np.nanmax(np.abs(np.where(np.isfinite(DS['rank_return']), DS['rank_return'], 0)), 1))[:10]
    print("days with the largest |rank return|:", [(str(DS["index"][i].date()), round(float(np.nanmax(np.abs(DS["rank_return"][i]))), 3)) for i in worst])
assert ratio_sd < 3, "rank returns are not clean (C1 gate): stop and report; do not train"

## 3. Rolling fits, both arms

In [ ]:
INFO, t_all, n_done, n_tot = [], time.time(), 0, len(ARMS) * len(SCHED)
for arm, mode in ARMS.items():
    SC = np.full(DS["ok"].shape, np.nan, np.float32)
    for j, f in enumerate(SCHED):
        tr_, va_, pr_ = (np.arange(*f[k]) for k in ("train", "valid", "predict"))
        tr_, va_ = tr_[DS["ok"][tr_].sum(1) > 0], va_[DS["ok"][va_].sum(1) > 0]
        seed_scores = []
        for s in CFG.seeds:
            p = CACHE / f"{arm}_fit{j:02d}_seed{s}.npz"
            if p.exists():
                z = np.load(p, allow_pickle=True); seed_scores.append(z["scores"]); info = json.loads(str(z["info"]))
            else:
                t1 = time.time()
                model, info = p7_model.fit(DS, tr_, va_, seed=s, mode=mode, gamma=CFG.gamma, epochs=CFG.epochs,
                                           patience=CFG.patience, lr=CFG.lr, chunk=CFG.chunk_days, device=DEVICE)
                sc = p7_model.predict_scores(model, DS, pr_, chunk=CFG.chunk_days)
                info["seconds"] = time.time() - t1
                np.savez(p, scores=sc, info=json.dumps(info)); seed_scores.append(sc)
            INFO.append({"arm": arm, "fit": j, "seed": s, "first_day": f["first_day"], **info})
        with np.errstate(invalid="ignore"):
            zs = [(x - np.nanmean(x, 1, keepdims=True)) / np.nanstd(x, 1, keepdims=True) for x in seed_scores]
            SC[pr_] = np.nanmean(np.stack(zs), 0)
        n_done += 1
        el = time.time() - t_all
        print(f"{arm} fit {j + 1}/{len(SCHED)} ({f['first_day']}): valid by seed "
              f"{[round(i['best_valid'], 2) for i in INFO[-len(CFG.seeds):]]}; {el / 60:.1f} min, "
              f"~{el / n_done * (n_tot - n_done) / 60:.0f} min left", flush=True)
    np.savez(CACHE / f"scores_{arm}.npz", scores=SC)
INFO = pd.DataFrame(INFO)
OUT = ROOT / "notebooks" / "Rank_Space" / ("01_synthetic" if SYNTHETIC else "01")
OUT.mkdir(parents=True, exist_ok=True)
INFO.to_csv(OUT / "fits.csv", index=False, float_format="%.5g")
if not SYNTHETIC:
    DIAG.to_csv(OUT / "shares_diagnostics.csv", index=False)
display(INFO.groupby(["arm", "fit"])[["best_valid", "train", "epochs"]].mean().unstack(0).round(3))
(OUT / "manifest.json").write_text(json.dumps({"files": sorted(p.name for p in OUT.iterdir()), "cache": str(CACHE), "device": DEVICE,
                                               "torch": torch.__version__, "p7": p7.P7_VERSION}, indent=2))